In [1]:
import torch
from matplotlib import pyplot

import importlib
import velocity_model
import estimate_velocity
importlib.reload(velocity_model)
importlib.reload(estimate_velocity)

L = 5e6
v0 = 1e3
Tr = L / (2 * v0)
origins = torch.tensor([2e6, 4e6], dtype=torch.int32)
h_tgrid = torch.linspace(0, 1.1, steps=11, dtype=torch.double)
g_xranges_l = torch.tensor([1e6, 4.5e6], dtype=torch.int32)
g_xranges_u = torch.tensor([1.1e6, 4.6e6], dtype=torch.int32)
mtrue = velocity_model.VelocityModel(L, True, origins, h_tgrid, g_xranges_l, g_xranges_u)

mtrue.h_log = torch.nn.Parameter(torch.log(torch.tensor([1, 1, 1.5, 2, 1.5, 1, 0.5, 0.1, 0.5, 1, 1])))
mtrue.g_log = torch.nn.Parameter(torch.log(torch.tensor([10, 0.1])))

In [2]:
coverage = 1e3
x = torch.linspace(0, L, steps=1000)
m = velocity_model.VelocityModel(L, True, origins, h_tgrid, g_xranges_l, g_xranges_u)
y_obs = torch.poisson(m.P(x) * coverage)
s_obs = torch.sqrt(y_obs)

torch.autograd.set_detect_anomaly(True)
l = velocity_model.VelocityModelLoss(x, y_obs, s_obs, m)
fit = estimate_velocity.FitModel(m, l)
fit.optimize(100)

/home/fgp/Versioned/OIST/replication/gaussian_process/venv/lib/python3.10/site-packages/torch/autograd/__init__.py:266: UserWarning: Error detected in MulBackward0. Traceback of forward call that caused the error:
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/fgp/Versioned/OIST/replication/gaussian_process/venv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/fgp/Versioned/OIST/replication/gaussian_process/venv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/fgp/Versioned/OIST/replication/gaussian_process/venv/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/home/fgp/Versioned/OIST/replication/gaussian_process/venv

RuntimeError: Function 'MulBackward0' returned nan values in its 1th output.